# NASDAQ-100 AI Stock Selection System

## Overview
This notebook implements a **machine learning-based stock selection system** for NASDAQ-100 stocks. The system uses 28 years of historical data (1995-2023) to train predictive models that identify the top-performing stocks for 2024.

### Key Features:
- **40 financial and technical factors** used as predictive features
- **7 different ML regression models** available for comparison
- **Historical training period**: 1995-2023
- **Prediction target**: 1-year forward stock returns
- **Output**: Top 10 stock picks ranked by predicted performance

### Methodology:
1. Load and prepare historical NASDAQ-100 factor data
2. Train machine learning models on historical returns
3. Predict expected returns for 2024
4. Rank and select top 10 stocks with highest predicted performance

### Dataset Features

The model uses **40 financial and technical factors** to predict stock performance. These features are categorized below:

#### 📊 **Fundamental Valuation Factors** (12 factors)
Ratios measuring company value relative to fundamentals:
- **Book to Price** - Book value / Market price
- **EBITDA to EV** - EBITDA / Enterprise Value
- **Cash Flow to EV** - Cash flow / Enterprise Value
- **FCF to EV** - Free Cash Flow / Enterprise Value
- **FCF to Price** - Free Cash Flow / Price
- **Cash Flow to Price** - Operating Cash Flow / Price
- **EBITDA to Price** - EBITDA / Market Cap
- **Earning to Price** (E/P ratio)
- **Net Current Assets to Price** - Working capital valuation
- **Operating Earnings to Price** - Operating income valuation
- **Pretax Income to Price** - Pre-tax earnings valuation
- **Sales to Price** - Revenue / Market Cap

#### 📈 **Momentum & Trend Factors** (11 factors)
Price momentum and trend indicators:
- **12M - 1M Price Momentum** - Medium-term momentum
- **15W to 36W Price Ratio** - Multi-week trend strength
- **1M Price Reversal** - Short-term mean reversion
- **39W Lag Return** - Long-term momentum
- **4W to 52W Oscillator** - Price oscillator
- **50D to 200D Price Ratio** - Golden cross indicator
- **52W Slope_AFL** - Annual trend slope
- **5D Price Reversal** - Very short-term reversal
- **60M Alpha** - Risk-adjusted outperformance
- **6M Alpha Change to 12M Alpha** - Alpha momentum
- **Closing Price to 52W High** - Distance from peak

#### 💰 **Cash Flow & Profitability Factors** (5 factors)
Measures of operational efficiency:
- **Cash Flow to Assets** - Asset productivity
- **FCF to Equity** - Free Cash Flow to shareholders
- **OCF to EV** - Operating Cash Flow / Enterprise Value
- **OCF to Price** - Operating Cash Flow / Market Cap
- **Sales to EV** - Revenue efficiency

#### 📉 **Volatility & Risk Factors** (6 factors)
Measures of price stability and risk:
- **12M Realized Volatility** - Annual volatility
- **1M Price High Low** - Monthly price range
- **1M Realized Volatility** - Monthly volatility
- **24M Residual Variance** - Idiosyncratic risk
- **52W Price High Low** - 52-week price range
- **60M CAPM Beta** - Market sensitivity
- **90D Coeff of Variation** - Relative volatility

#### 📊 **Technical & Market Indicators** (6 factors)
Technical analysis and market structure:
- **26W RSI** - Relative Strength Index (overbought/oversold)
- **Close to 260D Low** - Distance from yearly low
- **Log of Market Cap** - Company size (log scale)
- **Share Turnover** - Trading liquidity
- **Short Interest Ratio** - Days to cover short positions
- **Short Interest to ADTV** - Short interest relative to volume

---

### Data Structure
- **Training set**: 1995-2023 (28 years of historical data)
- **Prediction set**: 2024 data
- **Target variable**: **1Y return** (1-year forward stock returns)

In [1]:
# Import core data analysis libraries
import pandas as pd
import numpy as np

c:\Users\weekiang\.conda\envs\ml4t\lib\site-packages\numpy\_distributor_init.py:30: UserWarning: loaded more than 1 DLL from .libs:
c:\Users\weekiang\.conda\envs\ml4t\lib\site-packages\numpy\.libs\libopenblas.GK7GX5KEQ4F6UYO3P26ULGBQYHGQO7J4.gfortran-win_amd64.dll
c:\Users\weekiang\.conda\envs\ml4t\lib\site-packages\numpy\.libs\libopenblas.XWYDX2IKJW2NMTWSFYNGFUWKQU3LYTCZ.gfortran-win_amd64.dll
  warnings.warn("loaded more than 1 DLL from .libs:"


## 1. Data Preparation

### Load Historical Factor Data
Loading NASDAQ-100 stock data with 40 fundamental and technical factors from 1995 to 2024.

In [7]:
# Load historical NASDAQ-100 factor data
data = pd.read_csv("ml_factor_nq100_1995_2024_2025-04-20.csv", sep=";")

In [8]:
# Convert Date column to datetime format
data["Date"] = pd.to_datetime(data["Date"])

In [9]:
# Define training period
startdate = "1995-12-31"
enddate = "2023-12-31"

# Filter data for training period
bool_list = data["Date"].between(pd.to_datetime(startdate), pd.to_datetime(enddate))
X = data[bool_list]

In [10]:
# Extract features (columns 7-47: 40 financial/technical factors)
X = X.iloc[:, 7:47]

In [11]:
# Fill missing values with 0
X.fillna(0, inplace=True)

In [12]:
# Extract target variable (1-year forward returns)
yperf = data[bool_list]['1Y return']

In [13]:
# Fill missing values in target variable
yperf.fillna(0, inplace=True)

c:\Users\weekiang\.conda\envs\ml4t\lib\site-packages\pandas\core\series.py:4463: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  return super().fillna(


### Prepare 2024 Prediction Data
Extract features and ticker symbols for December 31, 2024 stocks.

In [14]:
# Extract 2024 data for prediction
X_2024 = data[data['Date'] == '2024-12-31']
X_2024 = X_2024.iloc[:, 7:47]

In [15]:
# Extract ticker symbols for 2024
tickers = data[data['Date'] == '2024-12-31'][['Date', '$ticker']].reset_index(drop=True)

## 2. Machine Learning Models

### Available Models
The system includes 7 different regression models with preprocessing pipelines:

1. **Linear Regression** - Simple linear model with PowerTransformer normalization
2. **ElasticNet** - Regularized linear model combining L1 and L2 penalties
3. **K-Nearest Neighbors** - Non-parametric model using 40 nearest neighbors
4. **Decision Tree** - Tree-based model with max depth of 20
5. **Random Forest** - Ensemble of 100 trees with max depth of 10
6. **Gradient Boosting** - Sequential ensemble with 100 estimators
7. **Support Vector Regression** - SVM with RBF kernel

In [16]:
# Import machine learning libraries
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PowerTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, ElasticNet
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
import pickle


def trainLinearModel(X_train, y_train):
    """Linear Regression with PowerTransformer normalization"""
    pl_linear = Pipeline([
        ('power_transformer', PowerTransformer()),
        ('linear', LinearRegression())
    ])
    pl_linear.fit(X_train, y_train)
    return pl_linear


def trainElasticNetModel(X_train, y_train):
    """ElasticNet Regression with L1/L2 regularization"""
    pl_ElasticNet = Pipeline([
        ('power_transformer', PowerTransformer()),
        ('elasticnet', ElasticNet(l1_ratio=0.00001))
    ])
    pl_ElasticNet.fit(X_train, y_train)
    return pl_ElasticNet


def trainKNeighborsModel(X_train, y_train):
    """K-Nearest Neighbors Regression with 40 neighbors"""
    pl_KNeighbors = Pipeline([
        ('power_transformer', PowerTransformer()),
        ('knn', KNeighborsRegressor(n_neighbors=40))
    ])
    pl_KNeighbors.fit(X_train, y_train)
    return pl_KNeighbors


def trainDecisionTreeModel(X_train, y_train):
    """Decision Tree Regression with max depth of 20"""
    pl_decTree = Pipeline([
        ('decision_tree', DecisionTreeRegressor(max_depth=20, random_state=42))
    ])
    pl_decTree.fit(X_train, y_train)
    return pl_decTree


def trainRandomForestModel(X_train, y_train):
    """Random Forest Regression ensemble with max depth of 10"""
    pl_rfregressor = Pipeline([
        ('random_forest', RandomForestRegressor(max_depth=10, random_state=41))
    ])
    pl_rfregressor.fit(X_train, y_train)
    return pl_rfregressor


def trainGradientBoostingModel(X_train, y_train):
    """Gradient Boosting Regression with 100 estimators"""
    pl_GradBregressor = Pipeline([
        ('gradient_boosting', GradientBoostingRegressor(
            n_estimators=100,
            learning_rate=0.1,
            max_depth=10,
            random_state=42,
            loss='squared_error'
        ))
    ])
    pl_GradBregressor.fit(X_train, y_train)
    return pl_GradBregressor


def trainSVMModel(X_train, y_train):
    """Support Vector Regression with RBF kernel"""
    pl_svm = Pipeline([
        ('power_transformer', PowerTransformer()),
        ('svr', SVR(kernel='rbf', C=100, gamma=0.1, epsilon=0.1))
    ])
    pl_svm.fit(X_train, y_train)
    return pl_svm

## 3. Train Model and Generate Predictions

### Select Your Model
Choose one of the available models by calling its training function:
- `trainLinearModel()` - Simple linear regression
- `trainElasticNetModel()` - Regularized linear model
- `trainKNeighborsModel()` - K-Nearest Neighbors (default)
- `trainDecisionTreeModel()` - Single decision tree
- `trainRandomForestModel()` - Ensemble of trees
- `trainGradientBoostingModel()` - Sequential boosting ensemble
- `trainSVMModel()` - Support Vector Machine

Currently using: **K-Nearest Neighbors** (best for finding similar patterns in historical data)

In [17]:
# Train all models and generate predictions
print("Training all 7 models...\n")

# Dictionary to store all models
models = {
    'Linear Regression': trainLinearModel(X, yperf),
    'ElasticNet': trainElasticNetModel(X, yperf),
    'K-Nearest Neighbors': trainKNeighborsModel(X, yperf),
    'Decision Tree': trainDecisionTreeModel(X, yperf),
    'Random Forest': trainRandomForestModel(X, yperf),
    'Gradient Boosting': trainGradientBoostingModel(X, yperf),
    'Support Vector Machine': trainSVMModel(X, yperf)
}

# Generate predictions for each model
predictions_df = tickers.copy()

for model_name, model in models.items():
    print(f"✓ {model_name} - Predictions generated")
    predictions = model.predict(X_2024)
    predictions_df[model_name] = predictions

print(f"\n{'='*80}")
print("All models trained successfully!")
print(f"{'='*80}")

Training all 7 models...

✓ Linear Regression - Predictions generated
✓ ElasticNet - Predictions generated
✓ K-Nearest Neighbors - Predictions generated
✓ Decision Tree - Predictions generated
✓ Random Forest - Predictions generated
✓ Gradient Boosting - Predictions generated
✓ Support Vector Machine - Predictions generated

All models trained successfully!


c:\Users\weekiang\.conda\envs\ml4t\lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\weekiang\.conda\envs\ml4t\lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
  File "c:\Users\weekiang\.conda\envs\ml4t\lib\subprocess.py", line 493, in run
    with Popen(*popenargs, **kwargs) as process:
  File "c:\Users\weekiang\.conda\envs\ml4t\lib\subprocess.py", line 858, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "c:\Users\weekiang\.conda\envs\ml4t\lib\subprocess.py", line 1311, in _execute_child
    hp, ht, pid, tid = _winapi.CreateProcess(executable, ar

In [18]:
# Display all predictions
print("\n" + "="*100)
print("PREDICTION RESULTS FOR ALL MODELS")
print("="*100 + "\n")

# Show all predictions
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.float_format', '{:.4f}'.format)

print(predictions_df.to_string(index=False))
print("\n")


PREDICTION RESULTS FOR ALL MODELS

      Date $ticker  Linear Regression  ElasticNet  K-Nearest Neighbors  Decision Tree  Random Forest  Gradient Boosting  Support Vector Machine
2024-12-31    AAPL             0.2178      0.4155               0.4324         0.2851         0.3091             0.3035                  0.3864
2024-12-31    ABNB            -0.0104     -0.0638              -0.0647        -0.0130        -0.0122            -0.0264                 -0.1439
2024-12-31    ADBE            -0.3416     -0.2103              -0.2187        -0.2218        -0.2435            -0.2802                 -0.2697
2024-12-31     ADI             0.0988      0.0378               0.0777         0.1038         0.0811             0.0822                  0.0405
2024-12-31     ADP             0.1991      0.2726               0.2018         0.2599         0.2550             0.2739                  0.3025
2024-12-31    ADSK             0.2816      0.3987               0.2463         0.2054         0.2228

In [19]:
# Calculate average prediction across all models
predictions_df['Average Score'] = predictions_df.iloc[:, 2:].mean(axis=1)

# Sort by average score and get top 10
top_stocks_by_average = predictions_df.sort_values('Average Score', ascending=False).head(10)

print("\n" + "="*100)
print("TOP 10 STOCKS BY AVERAGE PREDICTION ACROSS ALL MODELS")
print("="*100 + "\n")
print(top_stocks_by_average[['$ticker', 'Average Score', 'Linear Regression', 'K-Nearest Neighbors', 
                               'Random Forest', 'Gradient Boosting']].to_string(index=False))


TOP 10 STOCKS BY AVERAGE PREDICTION ACROSS ALL MODELS

$ticker  Average Score  Linear Regression  K-Nearest Neighbors  Random Forest  Gradient Boosting
    APP         4.9988             4.1538               3.7785         8.1239             7.5722
   MSTR         3.4914             2.9043               2.1422         4.6448             5.2741
   PLTR         3.0023             3.1043               3.5229         3.3598             2.9971
   AXON         1.3831             1.8097               1.3082         1.2996             1.3038
   NVDA         1.2203             1.2511               0.7368         1.7072             1.6232
   AVGO         1.0162             1.5772               0.9443         0.8887             0.8798
   TSLA         1.0067             1.5847               1.2582         0.6559             0.6417
   MRVL         0.9588             1.4604               0.8862         0.8564             0.8470
   NFLX         0.7937             1.0584               0.5609         

## 4. Individual Model Rankings

### Top 10 Stocks by Each Model
See which stocks each model predicts will perform best:

In [20]:
# Show top 10 stocks for each individual model
model_columns = ['Linear Regression', 'ElasticNet', 'K-Nearest Neighbors', 
                 'Decision Tree', 'Random Forest', 'Gradient Boosting', 'Support Vector Machine']

for model_name in model_columns:
    print(f"\n{'='*80}")
    print(f"TOP 10 STOCKS - {model_name.upper()}")
    print('='*80)
    
    top_10 = predictions_df.nlargest(10, model_name)[['$ticker', model_name]].reset_index(drop=True)
    top_10.index = top_10.index + 1  # Start index from 1
    print(top_10.to_string())
    print()


TOP 10 STOCKS - LINEAR REGRESSION
   $ticker  Linear Regression
1      APP             4.1538
2     PLTR             3.1043
3     MSTR             2.9043
4     AXON             1.8097
5     TSLA             1.5847
6     AVGO             1.5772
7     MRVL             1.4604
8     NVDA             1.2511
9     BKNG             1.2402
10     CEG             1.0707


TOP 10 STOCKS - ELASTICNET
   $ticker  ElasticNet
1      APP      3.0325
2     PLTR      2.3949
3     MSTR      2.1894
4     TSLA      1.5073
5     AXON      1.5010
6     AVGO      1.1511
7     MRVL      1.1128
8     BKNG      1.0595
9     NVDA      1.0091
10    NFLX      0.8713


TOP 10 STOCKS - K-NEAREST NEIGHBORS
   $ticker  K-Nearest Neighbors
1      APP               3.7785
2     PLTR               3.5229
3     MSTR               2.1422
4     AXON               1.3082
5     TSLA               1.2582
6     AVGO               0.9443
7     MRVL               0.8862
8     NVDA               0.7368
9     FTNT               0.

### Model Comparison Summary

Each model uses different algorithms to predict stock performance:

- **Linear/ElasticNet**: Fast linear models, good for straightforward relationships
- **K-Nearest Neighbors**: Pattern matching based on similar historical examples
- **Decision Tree**: Simple rule-based predictions
- **Random Forest**: Robust ensemble averaging multiple trees
- **Gradient Boosting**: Sequential learning, often most accurate
- **Support Vector Machine**: Advanced algorithm for complex patterns

**Average Score**: Consensus prediction across all 7 models - stocks appearing at the top across multiple models have stronger conviction.